In [1]:
!pip install imbalanced-learn
!pip install gensim
!pip install transformers datasets evaluate scikit-learn accelerate rouge_score sentencepiece seqeval



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score
import requests
import numpy as np
import pandas as pd
import re


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight


In [3]:
def download_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text.splitlines()



In [6]:
def process_lines(lines):
    data = []

    for sentence_id, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        tokens = line.split()
        for token in tokens:
            token = token.strip()
            if not token:
                continue

            # Hindi pattern: word\H=...
            if "\\H=" in token:
                match = re.match(r"^(.*?)\\H=.*$", token)
                if match:
                    word = match.group(1)
                    lang = "HI"

            elif token.endswith("\\E"):
                word = token.replace("\\E", "")
                lang = "EN"

            else:
                if re.fullmatch(r"[.,!?;:()\[\]{}\"'-]", token):
                    lang = "PUNCT"
                    word = token
                elif re.fullmatch(r"\d+", token):
                    lang = "NUM"
                    word = token
                else:
                    lang = "OTHER"
                    word = token

            word = re.sub(r"[^a-zA-Z\u0900-\u097F]", "", word)
            if not word:
                continue

            word = word.lower()
            data.append([sentence_id, word, lang])

    df = pd.DataFrame(data, columns=["sentence_id", "token", "lang"])
    return df


In [7]:
def assign_univ_tags(df):
    token_lang_counts = df.groupby(["token", "lang"]).size().unstack(fill_value=0)

    # Make sure both columns exist
    if "EN" not in token_lang_counts.columns:
        token_lang_counts["EN"] = 0
    if "HI" not in token_lang_counts.columns:
        token_lang_counts["HI"] = 0

    # Find tokens that appear in both HI and EN
    ambiguous_tokens = token_lang_counts[
        (token_lang_counts["EN"] > 0) & (token_lang_counts["HI"] > 0)
    ].index.tolist()

    print(f"🔍 Found {len(ambiguous_tokens)} ambiguous tokens appearing in both HI & EN.")
    print("📘 Example ambiguous tokens:", ambiguous_tokens[:10])

    # Assign UNIV tag
    df["lang"] = df.apply(
        lambda row: "UNIV" if row["token"] in ambiguous_tokens else row["lang"],
        axis=1
    )
    return df


In [8]:
dev_url = "https://cse.iitkgp.ac.in/resgrp/cnerg/qa/fire13translit/FIRE_Data/HindiEnglish_FIRE2013_AnnotatedDev.txt"
test_url = "https://cse.iitkgp.ac.in/resgrp/cnerg/qa/fire13translit/FIRE_Data/HindiEnglish_FIRE2013_Test_GT.txt"

dev_lines = download_data(dev_url)
test_lines = download_data(test_url)

dev_df = process_lines(dev_lines)
test_df = process_lines(test_lines)

dev_df = assign_univ_tags(dev_df)
test_df = assign_univ_tags(test_df)

print("✅ Processed dev set shape:", dev_df.shape)
print("✅ Processed test set shape:", test_df.shape)

print("\n🔹 Sample from dev_df:")
print(dev_df.head(15))


🔍 Found 10 ambiguous tokens appearing in both HI & EN.
📘 Example ambiguous tokens: ['are', 'in', 'india', 'life', 'matlab', 'me', 'my', 'one', 'paris', 'to']
🔍 Found 0 ambiguous tokens appearing in both HI & EN.
📘 Example ambiguous tokens: []
✅ Processed dev set shape: (3280, 3)
✅ Processed test set shape: (3358, 3)

🔹 Sample from dev_df:
    sentence_id    token lang
0             0     juda   HI
1             0     hoke   HI
2             0      bhi   HI
3             0       tu   HI
4             0   mujhme   HI
5             0    kahin   HI
6             0    baaki   HI
7             0      hai   HI
8             0    music   EN
9             0  release   EN
10            0     date   EN
11            1      iit   EN
12            1      jee   EN
13            1       ke   HI
14            1  results   EN


In [9]:
print(dev_df.columns)
print(dev_df["lang"].value_counts())


Index(['sentence_id', 'token', 'lang'], dtype='object')
lang
HI      2382
EN       840
UNIV      58
Name: count, dtype: int64


In [10]:
dev_df = dev_df[dev_df['lang'].isin(['HI', 'EN', 'UNIV'])].reset_index(drop=True)


print(dev_df.head(20))
print(f"Total tokens in DEV set: {len(dev_df)}")
print(f"Unique sentences: {dev_df['sentence_id'].nunique()}")
print(dev_df['lang'].value_counts())


    sentence_id     token lang
0             0      juda   HI
1             0      hoke   HI
2             0       bhi   HI
3             0        tu   HI
4             0    mujhme   HI
5             0     kahin   HI
6             0     baaki   HI
7             0       hai   HI
8             0     music   EN
9             0   release   EN
10            0      date   EN
11            1       iit   EN
12            1       jee   EN
13            1        ke   HI
14            1   results   EN
15            2  banarasi   HI
16            2      silk   EN
17            2    sarees   EN
18            3      main   HI
19            3      hoon   HI
Total tokens in DEV set: 3280
Unique sentences: 500
lang
HI      2382
EN       840
UNIV      58
Name: count, dtype: int64


In [11]:
X = dev_df['token']
y = dev_df['lang']

In [12]:
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(1, 3))
X_vec = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec,y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

          EN       0.93      0.78      0.85       163
          HI       0.93      0.98      0.95       479
        UNIV       0.86      0.86      0.86        14

    accuracy                           0.93       656
   macro avg       0.90      0.87      0.89       656
weighted avg       0.93      0.93      0.92       656



In [13]:
from sklearn.utils import resample

# Split by language
hi_df = dev_df[dev_df['lang'] == 'HI']
en_df = dev_df[dev_df['lang'] == 'EN']
univ_df = dev_df[dev_df['lang'] == 'UNIV']

# Get the target sample size (the largest class)
max_count = max(len(hi_df), len(en_df), len(univ_df))

# Upsample all minority classes
hi_up = resample(hi_df, replace=True, n_samples=max_count, random_state=42)
en_up = resample(en_df, replace=True, n_samples=max_count, random_state=42)
univ_up = resample(univ_df, replace=True, n_samples=max_count, random_state=42)

# Combine and shuffle
balanced_df = pd.concat([hi_up, en_up, univ_up]).sample(frac=1, random_state=42).reset_index(drop=True)

# Check new balance
print(balanced_df['lang'].value_counts())


lang
EN      2382
UNIV    2382
HI      2382
Name: count, dtype: int64


In [14]:
X = balanced_df['token']
y = balanced_df['lang']

tfidf = TfidfVectorizer(analyzer='char', ngram_range=(1, 4))
X_vec = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000, solver='lbfgs', C=2.0)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

          EN       0.99      0.99      0.99       476
          HI       0.99      0.99      0.99       477
        UNIV       1.00      1.00      1.00       477

    accuracy                           0.99      1430
   macro avg       0.99      0.99      0.99      1430
weighted avg       0.99      0.99      0.99      1430



In [15]:
import re

def predict_sentence(sentence, clf, vectorizer):
    # 1️⃣ Clean and tokenize input
    tokens = sentence.split()
    cleaned_tokens = [re.sub(r'[^a-zA-Z\u0900-\u097F]', '', t.lower()) for t in tokens]
    cleaned_tokens = [t for t in cleaned_tokens if t]  # remove empty tokens

    # 2️⃣ Transform using the same fitted TF-IDF vectorizer
    X_test = vectorizer.transform(cleaned_tokens)

    # 3️⃣ Predict using trained classifier
    preds = clf.predict(X_test)

    # 4️⃣ Return results as list of tuples
    return list(zip(tokens, preds))
print(predict_sentence("main hoon present today", clf, tfidf))
print(predict_sentence("aaj meeting cancel ho gayi because of rain", clf, tfidf))


[('main', 'HI'), ('hoon', 'HI'), ('present', 'EN'), ('today', 'HI')]
[('aaj', 'HI'), ('meeting', 'EN'), ('cancel', 'EN'), ('ho', 'HI'), ('gayi', 'HI'), ('because', 'EN'), ('of', 'EN'), ('rain', 'HI')]


In [16]:
X = dev_df['token']
y = dev_df['lang']

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5))
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

X_train_dense = X_train.toarray()

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_dense, y_train)

print("Class counts before SMOTE:\n", y_train.value_counts())
print("\nClass counts after SMOTE:\n", pd.Series(y_train_res).value_counts())

clf_smote = LogisticRegression(max_iter=1000, random_state=42)
clf_smote.fit(X_train_res, y_train_res)

y_pred_smote = clf_smote.predict(X_test.toarray())

print("\n📊 Classification Report (TF-IDF + SMOTE):")
print(classification_report(y_test, y_pred_smote))
print("Accuracy:", accuracy_score(y_test, y_pred_smote))


Class counts before SMOTE:
 lang
HI      1906
EN       672
UNIV      46
Name: count, dtype: int64

Class counts after SMOTE:
 lang
HI      1906
EN      1906
UNIV    1906
Name: count, dtype: int64

📊 Classification Report (TF-IDF + SMOTE):
              precision    recall  f1-score   support

          EN       0.93      0.85      0.88       168
          HI       0.95      0.98      0.96       476
        UNIV       1.00      1.00      1.00        12

    accuracy                           0.94       656
   macro avg       0.96      0.94      0.95       656
weighted avg       0.94      0.94      0.94       656

Accuracy: 0.9435975609756098


In [17]:
from gensim.models import FastText

sentences = dev_df.groupby('sentence_id')['token'].apply(list).tolist()

ft_model = FastText(
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    epochs=10
)

ft_model.build_vocab(sentences)
ft_model.train(sentences, total_examples=len(sentences), epochs=10)
ft_model.save("fasttext_stageA.model")

y = dev_df['lang']
classes = np.unique(y)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weight_dict = {cls: weight for cls, weight in zip(classes, class_weights)}
print("Class Weights:", class_weight_dict)

def get_vector(word):
    """Return FastText vector for word or zero vector if OOV"""
    if word in ft_model.wv:
        return ft_model.wv[word]
    else:
        return np.zeros(ft_model.vector_size)

X_vec = np.array([get_vector(str(w).lower()) for w in dev_df['token']])

X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

clf_fasttext = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf_fasttext.fit(X_train, y_train)

y_pred = clf_fasttext.predict(X_test)
print("\n📊 Classification Report (FastText + Class Weights):")
print(classification_report(y_test, y_pred))


Class Weights: {'EN': np.float64(1.3015873015873016), 'HI': np.float64(0.45899804086202073), 'UNIV': np.float64(18.850574712643677)}

📊 Classification Report (FastText + Class Weights):
              precision    recall  f1-score   support

          EN       0.29      0.56      0.38       168
          HI       0.46      0.13      0.21       476
        UNIV       0.05      0.83      0.10        12

    accuracy                           0.25       656
   macro avg       0.27      0.51      0.23       656
weighted avg       0.41      0.25      0.25       656



In [16]:
import transformers, torch
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)


c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 4.57.1
Torch version: 2.6.0+cu124


In [ ]:
# !pip install transformers datasets evaluate scikit-learn accelerate rouge_score sentencepiece

In [ ]:
# !pip install -U torch transformers


In [ ]:
# !pip install -U torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# import os
# os.kill(os.getpid(), 9)


In [20]:
import torch, transformers
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.6.0+cu124
Transformers version: 4.57.1
CUDA available: True


In [ ]:
# from google.colab import drive

# # Unmount if already mounted
# !fusermount -u /content/drive || true

# # Remove the folder if any leftover files
# !rm -rf /content/drive

# # Mount fresh
# drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import torch
import numpy as np

# -----------------------------
# Base directory for all stages
# -----------------------------
BASE_SAVE_DIR = "./ML_Models"

os.makedirs(BASE_SAVE_DIR, exist_ok=True)

STAGE_A_DIR = os.path.join(BASE_SAVE_DIR, "StageA_Models")
STAGE_B_DIR = os.path.join(BASE_SAVE_DIR, "StageB_Models")
STAGE_C_DIR = os.path.join(BASE_SAVE_DIR, "StageC_Models")

os.makedirs(STAGE_A_DIR, exist_ok=True)
os.makedirs(STAGE_B_DIR, exist_ok=True)
os.makedirs(STAGE_C_DIR, exist_ok=True)

# -----------------------------
# Check GPU availability
# -----------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


CUDA available: True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [17]:

import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [18]:
# ============================================
# Stage A — Language Identification (IndicBERTv2)
# ============================================

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import evaluate
import numpy as np

# -----------------------------
# Paths for Stage A
# -----------------------------
SAVE_DIR = STAGE_A_DIR
OUTPUT_DIR = os.path.join(SAVE_DIR, "StageA_Model")
SAVE_PATH = os.path.join(SAVE_DIR, "StageA_IndicBERTv2")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Prepare dataset
# -----------------------------
# Assuming dev_df is already loaded
df = dev_df[dev_df['lang'].isin(['HI', 'EN', 'UNIV'])].reset_index(drop=True)
grouped = df.groupby("sentence_id").agg({"token": list, "lang": list}).reset_index()

train_df, val_test = train_test_split(grouped, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(val_test, test_size=0.5, random_state=42)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

# -----------------------------
# Load IndicBERTv2 model
# -----------------------------
model_name = "ai4bharat/IndicBERTv2-MLM-only"
tokenizer = AutoTokenizer.from_pretrained(model_name)

label_list = ["EN", "HI", "UNIV"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# -----------------------------
# Tokenization with word alignment
# -----------------------------
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["token"], truncation=True, padding="max_length",
        max_length=64, is_split_into_words=True
    )
    labels = []
    for i, label_seq in enumerate(examples["lang"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = [-100 if w is None else label2id[label_seq[w]] for w in word_ids]
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# -----------------------------
# Metrics
# -----------------------------
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    true_preds = [
        [id2label[p] for (p,l) in zip(pred,label) if l!=-100]
        for pred,label in zip(preds,labels)
    ]
    true_labels = [
        [id2label[l] for (p,l) in zip(pred,label) if l!=-100]
        for pred,label in zip(preds,labels)
    ]
    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "f1": results["overall_f1"],
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "accuracy": results["overall_accuracy"]
    }

# -----------------------------
# Training
# -----------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# -----------------------------
# Final Evaluation + Save Model
# -----------------------------
metrics = trainer.evaluate(tokenized_dataset["test"])
print("Final Test Metrics:", metrics)

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Stage A model saved at: {SAVE_PATH}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-only and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 50/50 [00:00<?, ? examples/s]
C:\Users\garvi\AppData\Local\Temp\ipykernel_28248\3237300327.py:121: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,No log,0.156825,0.877551,0.928058,0.832258,0.969112
2,No log,0.175896,0.839344,0.853333,0.825806,0.959459
3,No log,0.166684,0.888889,0.900662,0.877419,0.969112
4,No log,0.202475,0.896774,0.896774,0.896774,0.965251
5,0.152000,0.205462,0.927632,0.946309,0.909677,0.978764
6,0.152000,0.203103,0.918033,0.933333,0.903226,0.974903
7,0.152000,0.213823,0.915033,0.927152,0.903226,0.974903


c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: HI seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: EN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: UNIV seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\garvi\AppData\

c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: HI seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: EN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: UNIV seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Final Test Metrics: {'eval_loss': 0.09476999193429947, 'eval_f1': 0.9039999999999999, 'eval_precision': 0.9112903225806451, 'eval_recall': 0.8968253968253969, 'eval_accuracy': 0.9794661190965093, 'eval_runtime': 0.8368, 'eval_samples_per_second': 59.751, 'eval_steps_per_second': 15.535, 'epoch': 7.0}
✅ Stage A model saved at: ./ML_Models\StageA_Models\StageA_IndicBERTv2


In [21]:
# path = "/content/drive/MyDrive/datasets/stageB_obfuscation_dataset_2000.csv"
path = "./stageB_obfuscation_dataset_2000.csv"
df = pd.read_csv(path)


print("Columns:", df.columns.tolist())
print("Total rows:", len(df))


print(df['obf_level'].value_counts())
print(df['meta_label'].value_counts())


df['obfuscated_text'] = df['obfuscated_text'].astype(str)
df['clean_text'] = df['clean_text'].astype(str)
df = df.dropna(subset=['obfuscated_text', 'clean_text']).reset_index(drop=True)
print("After dropna, rows:", len(df))


df.head(20)

Columns: ['sentence_id', 'clean_text', 'obfuscated_text', 'obf_level', 'meta_label']
Total rows: 2000
obf_level
low       800
medium    700
high      500
Name: count, dtype: int64
meta_label
SAFE      800
HATE      600
THREAT    600
Name: count, dtype: int64
After dropna, rows: 2000


,sentence_id,clean_text,obfuscated_text,obf_level,meta_label
0,1,"Transmission jam detected, postpone the meeting","7®ansm1ss1on jam detected , po5tpone the meeting",low,HATE
1,2,Code Alpha – maintain schedule,Cod3 Alph@ – maintain schedul3,medium,SAFE
2,3,"Multiple crossings detected near border side, ...","Multiple crossings detected near bo®der side ,...",low,SAFE
3,4,Recon reports possible infiltration near hill-1,Re(0n repo®ts poss!b13 infi|7ration ñea® #i1l - 1,medium,THREAT
4,5,"Sector 1 shows suspicious activity, verify","5Se(t0® @1# 5h0ws @$v6$pi(1ov5$ @a5c7iv17y# , ...",high,SAFE
5,6,All units remain in blackout; maintain radio s...,All uñ17s rem@!n iñ blackout ; maiñtain ®adio ...,medium,HATE
6,7,Establish perimeter and monitor the north ridge,Esta8l$h perimeter and monitor th n0®th ®idg,medium,HATE
7,8,Establish perimeter and monitor the north ridge,Establish perimeter @nd monitor t#e north ridge,medium,THREAT
8,9,"Send backup, radio is down","Send backup , radio is down",low,THREAT
9,10,"The source may shift position, keep eyes open","Th3 $0v®(e may $hif75 p05!t!0ñ , ke3p eye$ 0p3n",high,SAFE


In [ ]:
# ============================================
# Stage B — Obfuscation Normalization (mT5-base)
# ============================================

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

# -----------------------------
# Paths for Stage B
# -----------------------------
SAVE_DIR = STAGE_B_DIR
OUTPUT_DIR = os.path.join(SAVE_DIR, "mt5_small_stageB")
SAVE_PATH = os.path.join(SAVE_DIR, "mt5_small_stageB_best")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Step 1 — Prepare dataset
# -----------------------------
# Assuming df is already loaded with columns: obfuscated_text, clean_text
df = df[['obfuscated_text', 'clean_text']].dropna()

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, _ = train_test_split(temp_df, test_size=0.5, random_state=42)

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df.rename(columns={"obfuscated_text":"text","clean_text":"target"})
    ),
    "validation": Dataset.from_pandas(
        val_df.rename(columns={"obfuscated_text":"text","clean_text":"target"})
    )
})

# -----------------------------
# Step 2 — Load mT5 model
# -----------------------------
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.gradient_checkpointing_enable()

# Add PAD token if missing
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token":"<pad>"})
    model.resize_token_embeddings(len(tokenizer))

max_input_length = 64
max_target_length = 64

# -----------------------------
# Step 3 — Tokenization
# -----------------------------
def preprocess(batch):
    inputs = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_input_length
    )
    targets = tokenizer(
        batch["target"],
        truncation=True,
        padding="max_length",
        max_length=max_target_length
    )
    labels = [[tok if tok != tokenizer.pad_token_id else -100 for tok in seq]
              for seq in targets["input_ids"]]
    inputs["labels"] = labels
    return {k: np.array(v) for k,v in inputs.items()}

tokenized_dataset = dataset.map(preprocess, batched=True)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# -----------------------------
# Step 4 — Training
# -----------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=6,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    predict_with_generate=False,
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # use GPU if available
    load_best_model_at_end=True,
    save_total_limit=1,
    optim="adafactor",

)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

# -----------------------------
# Step 5 — Save best model
# -----------------------------
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Stage B model saved at: {SAVE_PATH}")


c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in 

Epoch,Training Loss,Validation Loss


In [20]:
# ============================================
# Stage B — Obfuscation Normalization (mT5-small, Quick Demo)
# ============================================

import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Adafactor
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Paths
# -----------------------------
SAVE_DIR = STAGE_B_DIR
OUTPUT_DIR = os.path.join(SAVE_DIR, "mt5_small_stageB_demo")
SAVE_PATH = os.path.join(SAVE_DIR, "mt5_small_stageB_demo_best")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Load dataset and subset for quick demo
# -----------------------------
df = df[['obfuscated_text', 'clean_text']].dropna()

# Take small subset for demo
train_df, temp_df = train_test_split(df, test_size=0.9, random_state=42)  # ~10% of data
val_df, _ = train_test_split(temp_df, test_size=0.5, random_state=42)

# -----------------------------
# Load model + tokenizer
# -----------------------------
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})
    model.resize_token_embeddings(len(tokenizer))

# Gradient checkpointing for memory saving
model.gradient_checkpointing_enable()
model.config.use_cache = False

# -----------------------------
# Tokenization
# -----------------------------
max_input_length = 48
max_target_length = 48

def preprocess(row):
    inputs = tokenizer(row["obfuscated_text"], truncation=True, padding="max_length",
                       max_length=max_input_length, return_tensors="pt")
    targets = tokenizer(row["clean_text"], truncation=True, padding="max_length",
                        max_length=max_target_length, return_tensors="pt")
    labels = targets["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return {"input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": labels.squeeze(0)}

train_data = [preprocess(row) for _, row in train_df.iterrows()]
val_data   = [preprocess(row) for _, row in val_df.iterrows()]

# -----------------------------
# PyTorch Dataset
# -----------------------------
class Seq2SeqDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

train_dataset = Seq2SeqDataset(train_data)
val_dataset   = Seq2SeqDataset(val_data)

# -----------------------------
# Collate function for batching
# -----------------------------
def collate_fn(batch):
    input_ids = pad_sequence([item["input_ids"] for item in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence([item["attention_mask"] for item in batch], batch_first=True, padding_value=0)
    labels = pad_sequence([item["labels"] for item in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=4, collate_fn=collate_fn)

# -----------------------------
# Optimizer
# -----------------------------
optimizer = Adafactor(model.parameters(), lr=3e-5, scale_parameter=True, relative_step=False)

# -----------------------------
# Training loop (2 epochs for demo)
# -----------------------------
num_epochs = 4
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {total_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss/len(val_loader):.4f}")

# -----------------------------
# Save model
# -----------------------------
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Stage B demo model saved at: {SAVE_PATH}")


c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Epoch 1/4, Training Loss: 28.3856
Epoch 1/4, Validation Loss: 21.3246
Epoch 2/4, Training Loss: 25.2239
Epoch 2/4, Validation Loss: 20.1487
Epoch 3/4, Training Loss: 25.0898
Epoch 3/4, Validation Loss: 19.2733
Epoch 4/4, Training Loss: 24.2749
Epoch 4/4, Validation Loss: 18.2113
✅ Stage B demo model saved at: ./ML_Models/StageB_Models\mt5_small_stageB_demo_best


In [23]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import evaluate
import os

# -----------------------------
# Paths for Stage C
# -----------------------------
SAVE_DIR = STAGE_C_DIR
OUTPUT_DIR = os.path.join(SAVE_DIR, "MuRIL_StageC")
SAVE_PATH = os.path.join(SAVE_DIR, "MuRIL_StageC_Best")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Step 1 — Prepare dataset
# -----------------------------
# Assuming df is already loaded with columns: clean_text, meta_label
df = df[['clean_text', 'meta_label']].dropna()
df['meta_label'] = df['meta_label'].astype(str)

# Label encoding
label2id = {label: idx for idx, label in enumerate(df["meta_label"].unique())}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["meta_label"].map(label2id)

# Split dataset
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
val_df, _ = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)

# Convert to HF Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["clean_text", "label"]].rename(columns={"clean_text":"text"}),
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        val_df[["clean_text", "label"]].rename(columns={"clean_text":"text"}),
        preserve_index=False
    )
})

# -----------------------------
# Step 2 — Load Model & Tokenizer
# -----------------------------
model_ckpt = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# -----------------------------
# Step 3 — Tokenization
# -----------------------------
max_length = 128
def preprocess(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

tokenized = dataset.map(preprocess, batched=True)

# -----------------------------
# Step 4 — Training Arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)

# -----------------------------
# Step 5 — Metrics
# -----------------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

# -----------------------------
# Step 6 — Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# -----------------------------
# Step 7 — Save Best Model
# -----------------------------
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"🎯 Stage C Training Complete. Model saved at: {SAVE_PATH}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 200/200 [00:00<00:00, 5659.07 examples/s]
C:\Users\garvi\AppData\Local\Temp\ipykernel_32064\438277427.py:113: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.092075,0.400000,0.228571
2,1.093000,1.088730,0.400000,0.228571
3,1.089800,1.088523,0.400000,0.228571


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


🎯 Stage C Training Complete. Model saved at: ./ML_Models/StageC_Models\MuRIL_StageC_Best


In [1]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    AutoModelForSeq2SeqLM, AutoModelForSequenceClassification
)

A_tok = AutoTokenizer.from_pretrained("./ML_Models/StageA_Models/StageA_IndicBERTv2")
A_model = AutoModelForTokenClassification.from_pretrained("./ML_Models/StageA_Models/StageA_IndicBERTv2").to("cuda")

B_tok = AutoTokenizer.from_pretrained("./ML_Models/StageB_Models/mt5_small_stageB_demo_best")
B_model = AutoModelForSeq2SeqLM.from_pretrained("./ML_Models/StageB_Models/mt5_small_stageB_demo_best").to("cuda")

C_tok = AutoTokenizer.from_pretrained("./ML_Models/StageC_Models/MuRIL_StageC_Best")
C_model = AutoModelForSequenceClassification.from_pretrained("./ML_Models/StageC_Models/MuRIL_StageC_Best").to("cuda")



# ---- Manual input ----
text = "k@l t0h I am c0m1ng meet m3 near #hill-1"

print("\n================= FULL PIPELINE =================")

# ---- Stage A ----
tokens = text.split()
encA = A_tok(tokens, is_split_into_words=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    logitsA = A_model(**encA).logits

id2label = A_model.config.id2label
langs = [id2label[i] for i in logitsA.argmax(dim=-1).squeeze().tolist()]

print("\n🔹 Stage A Token → Language:")
for t, l in zip(tokens, langs):
    print(f"{t:15s} → {l}")

# ---- Stage B ----
encB = B_tok(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    out_ids = B_model.generate(**encB, max_length=128)

clean_text = B_tok.decode(out_ids[0], skip_special_tokens=True)
print("\n🔹 Stage B Cleaned Text:")
print(clean_text)

# ---- Stage C ----
encC = C_tok(clean_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    logitsC = C_model(**encC).logits

label = C_model.config.id2label[logitsC.argmax().item()]
print("\n🔹 Stage C Final Label:", label)


c:\Users\garvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



================= FULL PIPELINE =================

🔹 Stage A Token → Language:
k@l             → HI
t0h             → HI
I               → EN
am              → EN
c0m1ng          → EN
meet            → HI
m3              → EN
near            → UNIV
#hill-1         → UNIV

🔹 Stage B Cleaned Text:
<extra_id_0>

🔹 Stage C Final Label: SAFE
